In [53]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.utils import resample



In [54]:
# ======================
# LOAD DATA
# ======================
df = pd.read_csv("C:\\Users\\dhamm\\Desktop\\project\\flight_data_2024_sample.csv")
df.columns = df.columns.str.strip().str.lower()



In [55]:
# ======================
# CLEAN DATA
# ======================
df = df.dropna(subset=['arr_delay', 'crs_dep_time'])
df.fillna(0, inplace=True)

df = df[(df['arr_delay'] < 300) & (df['arr_delay'] > -100)]


C:\Users\dhamm\AppData\Local\Temp\ipykernel_3016\4028546772.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(0, inplace=True)


In [56]:

# ======================
# TARGET
# ======================
df['delayed'] = df['arr_delay'].apply(lambda x: 1 if x > 15 else 0)



In [57]:
# ======================
# 🔥 BALANCE DATA (IMPORTANT FIX)
# ======================
df_major = df[df.delayed == 0]
df_minor = df[df.delayed == 1]

df_minor_upsampled = resample(
    df_minor,
    replace=True,
    n_samples=len(df_major),
    random_state=42
)

df = pd.concat([df_major, df_minor_upsampled])

print("Balanced dataset:")
print(df['delayed'].value_counts())


Balanced dataset:
delayed
0    7796
1    7796
Name: count, dtype: int64


In [58]:

# ======================
# FEATURE ENGINEERING
# ======================
df['flight_name'] = df['op_unique_carrier']
df['flight'] = df['flight_name'].astype('category').cat.codes
flight_classes = list(df['flight_name'].astype('category').cat.categories)

df['route_name'] = df['origin'] + "_" + df['dest']
df['route'] = df['route_name'].astype('category').cat.codes
route_classes = list(df['route_name'].astype('category').cat.categories)

df['hour'] = df['crs_dep_time'] // 100

df['is_peak'] = df['hour'].apply(
    lambda x: 1 if (7 <= x <= 10 or 17 <= x <= 21) else 0
)

df['is_weekend'] = df['day_of_week'].apply(
    lambda x: 1 if x in [6,7] else 0
)

def get_time_block(h):
    if h <= 6: return 0
    elif h <= 12: return 1
    elif h <= 18: return 2
    else: return 3

df['time_block'] = df['hour'].apply(get_time_block)


In [59]:

# ======================
# FEATURES
# ======================
features = [
    'flight',
    'route',
    'day_of_week',
    'crs_dep_time',
    'is_peak',
    'is_weekend',
    'time_block'
]

X = df[features]
y = df['delayed']


In [60]:

# ======================
# SPLIT
# ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [61]:

# ======================
# MODEL (STRONG + STABLE)
# ======================
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)


RandomForestClassifier(class_weight='balanced', max_depth=15,
                       min_samples_split=5, n_estimators=300, n_jobs=-1,
                       random_state=42)

In [62]:

# ======================
# EVALUATION
# ======================
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))



              precision    recall  f1-score   support

           0       0.90      0.78      0.84      1545
           1       0.81      0.91      0.86      1574

    accuracy                           0.85      3119
   macro avg       0.86      0.85      0.85      3119
weighted avg       0.86      0.85      0.85      3119



In [63]:
 #======================
# SAVE
# ======================
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(features, open("columns.pkl", "wb"))
pickle.dump(flight_classes, open("flight_classes.pkl", "wb"))
pickle.dump(route_classes, open("route_classes.pkl", "wb"))

print("✅ FINAL MODEL READY")

✅ FINAL MODEL READY


In [64]:
import pickle
print(pickle.load(open("columns.pkl","rb")))

['flight', 'route', 'day_of_week', 'crs_dep_time', 'is_peak', 'is_weekend', 'time_block']


In [66]:
df.columns

Index(['year', 'month', 'day_of_month', 'day_of_week', 'fl_date',
       'op_unique_carrier', 'op_carrier_fl_num', 'origin', 'origin_city_name',
       'origin_state_nm', 'dest', 'dest_city_name', 'dest_state_nm',
       'crs_dep_time', 'dep_time', 'dep_delay', 'taxi_out', 'wheels_off',
       'wheels_on', 'taxi_in', 'crs_arr_time', 'arr_time', 'arr_delay',
       'cancelled', 'cancellation_code', 'diverted', 'crs_elapsed_time',
       'actual_elapsed_time', 'air_time', 'distance', 'carrier_delay',
       'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay',
       'delayed', 'flight_name', 'flight', 'route_name', 'route', 'hour',
       'is_peak', 'is_weekend', 'time_block'],
      dtype='object')